# prototype clock value in jax
Make a working clock-value code with jax and jax-finufft that is fast enough!

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)

## bugs:
- Not yet written!

In [ ]:
# !pip install lightkurve
# !pip install jax

In [ ]:
import time
import jax
import jax.numpy as jnp
import lightkurve as lk
import matplotlib.pyplot as plt

In [ ]:
jax.config.update('jax_platform_name', 'cpu') # because bad Mac behavior?
jax.config.update("jax_enable_x64", True)

In [ ]:
def get_kepler_data(kic_id, exptime='long'):
    """
    ## Inputs:
    `kic_id`: Kepler ID (str)
    `exptime`: default exposure time, 'long'

    ## Outputs:
    Returns a 5-tuple:
    - `times`, `fluxes`, `errors`: light curve data
    - `delta_f`: frequency resolution, 1 / total observation time
    - `sampling_time`: median time between observations (in days)
    """    
    start = time.time()
    print("starting to obtain data for", kic_id)

    try:
        search_result = lk.search_lightcurve(kic_id, mission = 'Kepler', exptime=exptime)
        if len(search_result) < 1:
            msg = f"get_kepler_data(): no results for {kic_id} at this cadence"
            print(msg)
            update_error_message(kic_id, 'Kepler_long', msg)
            return jnp.nan, jnp.nan, jnp.nan, jnp.nan, jnp.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lk.search_lightcurve() failed for {kic_id } with {str(e)}")
        return jnp.nan, jnp.nan, jnp.nan, jnp.nan, jnp.nan

    try:
        lc_collection = search_result.download_all()
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return jnp.nan, jnp.nan, jnp.nan, jnp.nan, jnp.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return jnp.nan, jnp.nan, jnp.nan, jnp.nan, jnp.nan

    try:
        lc = lc_collection.stitch()
        #print("get_kepler_data(): minimum time value", jnp.min(lc.time.value), jnp.min(lc.time), lc.time)
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return jnp.nan, jnp.nan, jnp.nan, jnp.nan, jnp.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return jnp.nan, jnp.nan, jnp.nan, jnp.nan, jnp.nan

    # unpack, remove bad data, and reorder
    times, fluxes, errors = lc.time.value, lc.flux.value, lc.flux_err.value
    good = jnp.isfinite(times) & jnp.isfinite(fluxes) & jnp.isfinite(errors)
    times, fluxes, errors = times[good], fluxes[good], errors[good]
    idx = jnp.argsort(times)
    times, fluxes, errors = times[idx], fluxes[idx], errors[idx]

    delta_f = (1/(times[-1] - times[0]))
    sampling_time= jnp.median(jnp.diff(times))
    print("get_kepler_data() took", time.time() - start, "seconds")

    return times, fluxes, errors, delta_f, sampling_time

In [ ]:
# get data on a good example

ts, ys, errs, deltaf, deltat = get_kepler_data("KIC005285607")
print(ts.shape, ys.shape, errs.shape, deltaf, deltat)

In [ ]:
# check the data

f = plt.figure(figsize=(9, 3))
plt.scatter(ts, ys, s=1, c="k", marker=".")

In [ ]:
# start by solving this problem with standard (non-finufft) methods

def design_matrix(om, t, Mmax=16):
    ms1, ms2 = jnp.arange(Mmax), jnp.arange(1, Mmax)
    return jnp.concat((jnp.cos(ms1[None, :] * om * t[:, None]),
                      jnp.sin(ms2[None, :] * om * t[:, None])), axis=1), \
            jnp.concat((ms1, ms2))
                            

In [ ]:
# check the design matrix

foo, ms = design_matrix(0.3445, ts)
print(foo.shape, ys.shape, ms.shape)
print(ms)

In [ ]:
# time to get the clock value
# bug: doesn't regularize away the frequencies that are above Nyquist
# bug: doesn't use jax_finufft

def clock_value(om, t, y, iv):
    X, m = design_matrix(om, t)
    A = X.T @ (iv[:, None] * X)
    b = X.T @ (iv * ys)
    pars = jnp.linalg.solve(A, b)
    return jnp.sum(om ** 2 * m **2 * pars ** 2)

In [ ]:
# now laboriously and slowly do all frequencies
# bug: this needs to make use of jax_finufft

ivars = 1. / errs ** 2
smallest = 2. * jnp.pi * 0.1 # smallest angular frequency to consider MAGIC 10-day period
nyquist = jnp.pi / deltat

best_value = 0.
best_omega = 0.
oms = jnp.arange(smallest, nyquist, jnp.pi * deltaf)
values = jax.vmap(clock_value, in_axes=(0, None, None, None))(oms, ts, ys, ivars)
print(oms.shape, values.shape)

In [ ]:
# THIS BLOCK CAUSES MY KERNEL TO DIE, NO MATTER HOW I ARRANGE IT
# many people died to try to get around this one.

best_omega = oms[jnp.argmax(values)]
print(best_omega)

In [ ]:
assert False

In [ ]:
# get best frequency
# BUG: This should use the parabola trick

@jax.jit
def get_best_peak(xs, ys):
    ii = jnp.argmax(ys)
    return xs[ii]

best_omega = get_best_peak(oms, values)
print(best_omega)
assert False

In [ ]:
# plt.plot(oms, values, "k.")
# plt.plot(oms, values, "k-")

In [ ]:
# look at best frequency
# bug: this needs to do the parabola trick first to refine

best_T = 2. * jnp.pi / best_omega
f = plt.figure(figsize=(9, 3))
plt.scatter(ts % best_T, ys, s=1, c="k", marker=".")